## Load 

In [ ]:
import torch
import pickle
import os
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Path to saved model
model_path = './saved_model'

# Load model and tokenizer
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model.to(device)
model.eval()

# Load label encoder
with open(os.path.join(model_path, 'label_encoder.pkl'), 'rb') as f:
    mlb = pickle.load(f)



## String to evaluate - title of the law

In [ ]:
title = 'Regolamento per il commercio di prodotti agricoli in europa'


## Run model

In [ ]:
# Tokenize input
encoded_dict = tokenizer.encode_plus(
    title,
    add_special_tokens=True,
    max_length=512,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_tensors='pt'
)

input_ids = encoded_dict['input_ids'].to(device)
attention_mask = encoded_dict['attention_mask'].to(device)

# Predict
with torch.no_grad():
    outputs = model(input_ids, attention_mask=attention_mask)
    logits = outputs.logits

# Apply sigmoid and 0.5 threshold for multi-label
probs = torch.sigmoid(logits).cpu().numpy()[0]
predictions = (probs >= 0.5).astype(int)



## Derive domain

In [ ]:
# Decode labels
predicted_labels = mlb.inverse_transform(predictions.reshape(1, -1))
print(f'Law Title: {title}')
print(f'Predicted Domains: {predicted_labels[0]}')

